# Customers and Products Analysis

Loading `stores.db` (SQLite) with pandas for analysis.

In [12]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("stores.db")

## List of Tables

In [13]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';", conn
)
tables

,name
0,customers
1,employees
2,offices
3,orderdetails
4,orders
5,payments
6,productlines
7,products


## Database Overview

Before diving deeper into the analysis, let's get a high-level snapshot of the database: how many columns (attributes) each table has and how many rows of data it contains. This gives a quick sense of the size and structure of each table before querying them for insights.

In [14]:
query = """
SELECT 'Customers' AS table_name,
       (SELECT count(*) FROM pragma_table_info('customers')) AS number_of_attributes,
       count(*) AS number_of_rows
FROM customers

UNION ALL

SELECT 'Products',
       (SELECT count(*) FROM pragma_table_info('products')),
       count(*)
FROM products

UNION ALL

SELECT 'ProductLines',
       (SELECT count(*) FROM pragma_table_info('productlines')),
       count(*)
FROM productlines

UNION ALL

SELECT 'Orders',
       (SELECT count(*) FROM pragma_table_info('orders')),
       count(*)
FROM orders

UNION ALL

SELECT 'OrderDetails',
       (SELECT count(*) FROM pragma_table_info('orderdetails')),
       count(*)
FROM orderdetails

UNION ALL

SELECT 'Payments',
       (SELECT count(*) FROM pragma_table_info('payments')),
       count(*)
FROM payments

UNION ALL

SELECT 'Employees',
       (SELECT count(*) FROM pragma_table_info('employees')),
       count(*)
FROM employees

UNION ALL

SELECT 'Offices',
       (SELECT count(*) FROM pragma_table_info('offices')),
       count(*)
FROM offices
"""

df_summary = pd.read_sql_query(query, conn)
df_summary

,table_name,number_of_attributes,number_of_rows
0,Customers,13,122
1,Products,9,110
2,ProductLines,4,7
3,Orders,7,326
4,OrderDetails,5,2996
5,Payments,4,273
6,Employees,8,23
7,Offices,9,7


## Priority Products for Restocking

Identify products that are running low on stock AND are strong sellers among that low-stock group, so restocking effort goes to the products that matter most:
1. **Low stock** — a correlated subquery estimates how many times over the current stock has been sold (rounded down to 2 decimals), ranked to find the top 10 lowest-stock products. Returned/negative order lines (`quantityOrdered < 0`) are excluded so they can't distort the ratio. Products with `quantityInStock = 0` are treated as maximum priority (forced to the top with a sentinel value) instead of being dropped, since a stockout with ongoing demand is the most urgent case, not an edge case to ignore.
2. **Product performance** — restricted to those top-10 low-stock products, total revenue (`quantityOrdered * priceEach`) is computed per product to rank the top 10 best sellers among them.
3. **Priority list** — the final query looks up `productName` and `productLine` for the products that made it through both filters (best sellers that are also running low).

> **Known limitation:** the `products` table has no `discontinued`/status column, so a `quantityInStock = 0` product that's genuinely out of stock (needs urgent restocking) can't be distinguished here from one that's simply discontinued (doesn't need restocking at all). With this schema, both get treated as top priority.

In [15]:
query_priority_restock = """
WITH low_stock AS (
    SELECT p.productCode,
           p.productName,
           p.productLine,
           (
               SELECT CASE
                          WHEN p.quantityInStock = 0 THEN 99999
                          ELSE FLOOR((SUM(o.quantityOrdered) * 1.0 / p.quantityInStock) * 100) / 100.0
                      END
               FROM orderdetails AS o
               WHERE o.productCode = p.productCode
                 AND o.quantityOrdered >= 0
           ) AS low_stock
    FROM products AS p
    WHERE p.quantityInStock >= 0
    ORDER BY low_stock DESC
    LIMIT 10
),

product_performance AS (
    SELECT o.productCode,
           SUM(o.quantityOrdered * o.priceEach) AS product_performance
    FROM orderdetails AS o
    WHERE o.productCode IN (SELECT productCode FROM low_stock)
    GROUP BY o.productCode
    ORDER BY product_performance DESC
    LIMIT 10
)

SELECT productName, productLine
FROM products AS p
WHERE productCode IN (SELECT productCode FROM product_performance)
"""

df_priority_restock = pd.read_sql_query(query_priority_restock, conn)
df_priority_restock

,productName,productLine
0,1968 Ford Mustang,Classic Cars
1,1911 Ford Town Car,Vintage Cars
2,1928 Mercedes-Benz SSK,Vintage Cars
3,1960 BSA Gold Star DBD34,Motorcycles
4,1997 BMW F650 ST,Motorcycles
5,1996 Peterbilt 379 Stake Bed with Outrigger,Trucks and Buses
6,1928 Ford Phaeton Deluxe,Vintage Cars
7,2002 Yamaha YZR M1,Motorcycles
8,F/A 18 Hornet 1/72,Planes
9,Pont Yacht,Ships


## Profit per Customer

Join `orders`, `orderdetails`, and `products` to bring customer and product info together, then compute each customer's total profit: `SUM(quantityOrdered * (priceEach - buyPrice))`.

In [16]:
query_customer_profit = """
SELECT o.customerNumber,
       SUM(od.quantityOrdered * (od.priceEach - pr.buyPrice)) AS profit
FROM orders AS o
JOIN orderdetails AS od
  ON o.orderNumber = od.orderNumber
JOIN products AS pr
  ON pr.productCode = od.productCode
GROUP BY o.customerNumber
"""

df_customer_profit = pd.read_sql_query(query_customer_profit, conn)
df_customer_profit

,customerNumber,profit
0,103,10063.80
1,112,31312.72
2,114,70311.07
3,119,60875.30
4,121,41391.52
...,...,...
93,486,33598.57
94,487,17230.12
95,489,10868.04
96,495,25244.69


## Top 5 VIP Customers

Reuse the customer-profit query as a CTE, then join it to `customers` to pull contact and location details for the 5 most profitable customers.

In [17]:
query_vip_customers = """
WITH customer_profit AS (
    SELECT o.customerNumber,
           SUM(od.quantityOrdered * (od.priceEach - p.buyPrice)) AS profit
    FROM products AS p
    JOIN orderdetails AS od
      ON p.productCode = od.productCode
    JOIN orders AS o
      ON o.orderNumber = od.orderNumber
    GROUP BY o.customerNumber
)

SELECT c.contactLastName,
       c.contactFirstName,
       c.city,
       c.country,
       cp.profit
FROM customers AS c
JOIN customer_profit AS cp
  ON cp.customerNumber = c.customerNumber
ORDER BY cp.profit DESC
LIMIT 5
"""

df_vip_customers = pd.read_sql_query(query_vip_customers, conn)
df_vip_customers

,contactLastName,contactFirstName,city,country,profit
0,Freyre,Diego,Madrid,Spain,326519.66
1,Nelson,Susan,San Rafael,USA,236769.39
2,Young,Jeff,NYC,USA,72370.09
3,Ferguson,Peter,Melbourne,Australia,70311.07
4,Labrune,Janine,Nantes,France,60875.30


## Top 5 Least-Engaged Customers

Same query as above, but sorted ascending to surface the 5 customers generating the least profit.

In [19]:
query_least_engaged_customers = """
WITH customer_profit AS (
    SELECT o.customerNumber,
           SUM(od.quantityOrdered * (od.priceEach - p.buyPrice)) AS profit
    FROM products AS p
    JOIN orderdetails AS od
      ON p.productCode = od.productCode
    JOIN orders AS o
      ON o.orderNumber = od.orderNumber
    GROUP BY o.customerNumber
)

SELECT c.contactLastName,
       c.contactFirstName,
       c.city,
       c.country,
       cp.profit
FROM customers AS c
JOIN customer_profit AS cp
  ON cp.customerNumber = c.customerNumber
ORDER BY cp.profit ASC
LIMIT 5
"""

df_least_engaged_customers = pd.read_sql_query(query_least_engaged_customers, conn)
df_least_engaged_customers

,contactLastName,contactFirstName,city,country,profit
0,Young,Mary,Glendale,USA,2610.87
1,Taylor,Leslie,Brickhaven,USA,6586.02
2,Ricotti,Franco,Milan,Italy,9532.93
3,Schmitt,Carine,Nantes,France,10063.80
4,Smith,Thomas,London,UK,10868.04


## Average Customer Profit

Reuse the customer-profit CTE to compute the average profit across all customers, as a baseline to compare individual customers against.

In [20]:
query_avg_customer_profit = """
WITH customer_profit AS (
    SELECT o.customerNumber,
           SUM(od.quantityOrdered * (od.priceEach - p.buyPrice)) AS profit
    FROM products AS p
    JOIN orderdetails AS od
      ON p.productCode = od.productCode
    JOIN orders AS o
      ON o.orderNumber = od.orderNumber
    GROUP BY o.customerNumber
)

SELECT AVG(profit) AS avg_customer_profit
FROM customer_profit
"""

df_avg_customer_profit = pd.read_sql_query(query_avg_customer_profit, conn)
df_avg_customer_profit

,avg_customer_profit
0,39039.594388


In [21]:
conn.close()